# LeetCode 536 — Construct Binary Tree from String bằng Python

Link đề: https://leetcode.com/problems/construct-binary-tree-from-string/ · Độ khó: **Medium** · Chủ đề: String, Stack, Tree, Binary Tree, Depth-First Search

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def str2tree(self, s: str) -> Optional[TreeNode]:
        ...
```

Ghi chú: đây là bài **Premium** (có khóa). Đề trong notebook được viết theo hiểu biết về bài, nên hãy **đối chiếu lại đề gốc nếu bạn có tài khoản Premium**. Notebook tự dựng `TreeNode`, hàm `build_tree` (list kiểu LeetCode -> cây), hàm `tree_to_list` (cây -> list) và hàm tuần tự hóa `tree2str` (cây -> chuỗi) để kiểm tra kết quả. Các ví dụ ngoài ví dụ 1-3 là do notebook tự đặt (đáp án đã kiểm chứng bằng code) - cần thì đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho một **xâu** mô tả cây nhị phân theo kiểu "số, rồi (cây con trái), rồi (cây con phải)" và dựng lại **cây** thật bằng các nút `TreeNode`.
2. Nắm các công cụ Python cần dùng: cắt xâu (`s[i:j]`), `str.find`, `str.isdigit`, `int("-4")`, đọc **một số nhiều chữ số có dấu âm** bằng con trỏ `i, j`, `list` làm **ngăn xếp (stack)**, đệ quy với `nonlocal`, và giới hạn đệ quy `sys.setrecursionlimit`.
3. Đi từ **cắt xâu rồi đệ quy** (đếm ngoặc để tìm khối con) -> **đệ quy xuống đúng chỗ bằng một con trỏ** (recursive descent) -> **một lượt quét với ngăn xếp** -> **tính trước cặp ngoặc khớp nhau rồi đệ quy theo khoảng chỉ số**.
4. Hiểu hai mấu chốt: (a) `(` nghĩa là **đi xuống một cấp**, `)` nghĩa là **quay lên**; khối ngoặc **đầu tiên** của một nút là con **trái**, khối **thứ hai** là con **phải**; (b) chỉ cần đọc mỗi ký tự **một lần** nếu ta nhớ được **đường đi từ gốc đến nút hiện tại**.
5. Tránh các lỗi kinh điển: đọc từng ký tự thay vì cả số (mất số nhiều chữ số và dấu `-`), luôn gán vào `left` (mất con phải), quên `pop` khi gặp `)`, quên bỏ qua `)` trong đệ quy, và quên trường hợp xâu rỗng.

## 2. Đề bài & ví dụ

Bạn cần dựng một cây nhị phân từ một xâu gồm **số nguyên và dấu ngoặc**.

Toàn bộ xâu mô tả **một** cây nhị phân. Xâu gồm một **số nguyên** (giá trị của nút gốc), theo sau là **không, một hoặc hai** cặp ngoặc. Mỗi cặp ngoặc chứa một cây con có cấu trúc **giống hệt** (tức là lại là "số nguyên + các cặp ngoặc"). Luôn dựng con **trái** trước nếu nó tồn tại: cặp ngoặc đầu tiên là con trái, cặp ngoặc thứ hai là con phải.

**Ví dụ 1**

- Input: `s = "4(2(3)(1))(6(5))"`
- Output: `[4,2,6,3,1,5]`

**Ví dụ 2**

- Input: `s = "4(2(3)(1))(6(5)(7))"`
- Output: `[4,2,6,3,1,5,7]`

**Ví dụ 3**

- Input: `s = "-4(2(3)(1))(6(5)(7))"`
- Output: `[-4,2,6,3,1,5,7]`

(Kết quả viết theo kiểu LeetCode: duyệt theo **từng tầng**, từ trái sang phải. Trong notebook ta kiểm tra bằng hàm `tree_to_list`.)

**Ràng buộc:**

- `0 <= s.length <= 3 * 10^4`
- `s` chỉ gồm chữ số, `'('`, `')'` và `'-'`.
- Mọi giá trị trong cây có trị tuyệt đối **không vượt quá** `2^30`.

Chú ý quan trọng:

- Xâu **rỗng** `""` là hợp lệ: kết quả là cây rỗng (`None`).
- Giá trị có thể **nhiều chữ số** (ví dụ `1073741824`) và có thể **âm** (`-4`).
- Đề không nói rõ cây có con **phải mà không có con trái** thì viết thế nào. Trong bài này nếu một nút chỉ có một con thì cặp ngoặc đó luôn là con **trái**. Để chắc ăn, các lời giải của ta **chịu được** cả kiểu viết `"1()(2)"` (cặp ngoặc rỗng `()` = con trống), đây là kiểu viết của bài LeetCode 606; nếu có Premium hãy đối chiếu lại đề gốc.

In [1]:
from typing import Optional, List
from collections import deque
import random
import sys
import time

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ list kiểu LeetCode (duyệt theo tầng, None là nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> list kiểu LeetCode (cắt None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


def tree2str(root):
    """Tuần tự hóa cây thành chuỗi (kiểu LeetCode 606): số, (trái), (phải).
    Nếu chỉ có con phải thì con trái viết là "()". Dùng ngăn xếp tường minh để không lo đệ quy sâu."""
    out = []
    stack = [root] if root else []
    while stack:
        x = stack.pop()
        if isinstance(x, str):
            out.append(x)
            continue
        out.append(str(x.val))
        if x.right:
            stack += [")", x.right, "("]       # đẩy nhóm phải trước để nó được lấy ra SAU
        if x.left or x.right:
            stack.append(")")
            if x.left:
                stack.append(x.left)
            stack.append("(")
    return "".join(out)


def same_tree(a, b):
    """So sánh hai cây có cùng hình dạng và cùng giá trị hay không (không đệ quy)."""
    stack = [(a, b)]
    while stack:
        x, y = stack.pop()
        if x is None and y is None:
            continue
        if x is None or y is None or x.val != y.val:
            return False
        stack.append((x.left, y.left))
        stack.append((x.right, y.right))
    return True


examples = [
    ("4(2(3)(1))(6(5))", [4, 2, 6, 3, 1, 5]),
    ("4(2(3)(1))(6(5)(7))", [4, 2, 6, 3, 1, 5, 7]),
    ("-4(2(3)(1))(6(5)(7))", [-4, 2, 6, 3, 1, 5, 7]),
]
for s, expected in examples:
    t = build_tree(expected)
    assert tree2str(t) == s          # kiểm tra: tuần tự hóa cây mẫu cho đúng xâu đề cho
    print(repr(s), "-> kỳ vọng", expected)

'4(2(3)(1))(6(5))' -> kỳ vọng [4, 2, 6, 3, 1, 5]
'4(2(3)(1))(6(5)(7))' -> kỳ vọng [4, 2, 6, 3, 1, 5, 7]
'-4(2(3)(1))(6(5)(7))' -> kỳ vọng [-4, 2, 6, 3, 1, 5, 7]


## 3. Kiến thức nền

### 3.1. Nút cây `TreeNode` và cách vẽ cây

Mỗi nút có `val`, `left`, `right`. Con trỏ không có con thì là `None`. Ví dụ 1 của đề tương ứng với cây:

```
        4
      /   \
     2     6
    / \   /
   3   1 5
```

Ta tự dựng cây này **bằng tay** để thấy `TreeNode` hoạt động thế nào.

In [2]:
a = TreeNode(4)
a.left = TreeNode(2)
a.right = TreeNode(6)
a.left.left = TreeNode(3)
a.left.right = TreeNode(1)
a.right.left = TreeNode(5)

print(tree_to_list(a))           # list theo từng tầng
print(tree2str(a))               # chính là xâu của ví dụ 1
print(a.left.right.val, a.right.right)   # 1 và None (nút 6 không có con phải)

[4, 2, 6, 3, 1, 5]
4(2(3)(1))(6(5))
1 None


### 3.2. Định dạng xâu: ngữ pháp "số + (trái) + (phải)"

Ta có thể mô tả định dạng bằng một "công thức đệ quy" (gọi là **ngữ pháp**):

```
Cây  :=  số  [ "(" Cây ")"  [ "(" Cây ")" ] ]
```

Đọc là: một cây = một số, **có thể** theo sau bởi một cặp ngoặc chứa một cây (con trái), và **có thể** theo sau nữa bởi cặp ngoặc thứ hai chứa một cây (con phải). Dấu `[ ]` nghĩa là "tùy chọn".

Ví dụ nhỏ:

- `"7"` : nút `7`, không có con.
- `"7(8)"` : nút `7` có con trái `8`.
- `"7(8)(9)"` : nút `7` có con trái `8`, con phải `9`.
- `"7(8(1))(9)"` : con trái của `7` là cây `8(1)` (nút `8` có con trái `1`).

Điều thú vị: **vế phải của ngữ pháp lại chứa chính "Cây"**. Vì vậy bài này tự nhiên nghĩ tới **đệ quy**.

In [3]:
for s in ["7", "7(8)", "7(8)(9)", "7(8(1))(9)", "7()(9)"]:
    print(f"{s!r:14} ->", end=" ")
    # Chưa có lời giải, nên dựng bằng tay từ list để thấy hình dạng:
    manual = {"7": [7], "7(8)": [7, 8], "7(8)(9)": [7, 8, 9], "7(8(1))(9)": [7, 8, 9, 1], "7()(9)": [7, None, 9]}[s]
    t = build_tree(manual)
    print(manual, "| tuần tự hóa lại:", tree2str(t))

'7'            -> [7] | tuần tự hóa lại: 7
'7(8)'         -> [7, 8] | tuần tự hóa lại: 7(8)
'7(8)(9)'      -> [7, 8, 9] | tuần tự hóa lại: 7(8)(9)
'7(8(1))(9)'   -> [7, 8, 9, 1] | tuần tự hóa lại: 7(8(1))(9)
'7()(9)'       -> [7, None, 9] | tuần tự hóa lại: 7()(9)


Dòng cuối cho thấy `"7()(9)"`: nút `7` **chỉ có con phải** nên con trái phải viết `()` mới biết `9` là con phải. Đó là kiểu viết của LeetCode 606; ta sẽ chịu được nó cho an toàn.

### 3.3. Xâu trong Python: chỉ số, cắt lát, `find`, `isdigit`, `int`

- `s[i]` lấy ký tự thứ `i` (từ `0`). `s[i:j]` lấy các ký tự từ `i` đến `j-1` (không gồm `j`). `len(s)` là độ dài.
- `s.find("(")` trả về vị trí xuất hiện **đầu tiên** của `"("`, hoặc `-1` nếu không có.
- `c.isdigit()` cho `True` nếu ký tự `c` là chữ số `0..9`. Chú ý: `"-".isdigit()` là `False`!
- `int("12")` là `12`, `int("-4")` là `-4`, nhưng `int("")` **báo lỗi** `ValueError`.

In [4]:
s = "4(2(3)(1))(6(5))"
print(s[0], s[1], s[-1])        # ký tự đầu, thứ hai, cuối
print(s[2:5])                   # "2(3"
print(s.find("("), s.find("x"))
print("5".isdigit(), "-".isdigit(), "(".isdigit())
print(int("12"), int("-4"), int("1073741824"))
try:
    int("")
except ValueError as e:
    print("int('') ->", type(e).__name__)

4 ( )
2(3
1 -1
True False False
12 -4 1073741824
int('') -> ValueError


### 3.4. Đọc **một số nhiều chữ số** bằng hai con trỏ `i` và `j`

Đây là kỹ năng cốt lõi của mọi bài "phân tích cú pháp" (parsing). Đứng ở vị trí `i` là ký tự đầu của số (một chữ số hoặc dấu `-`). Cho `j` chạy tiếp **chừng nào còn là chữ số**; khi dừng, số nằm ở `s[i:j]`.

Chi tiết quan trọng: nếu `s[i]` là `'-'` thì nó **không phải chữ số** nên ta bắt đầu `j` từ `i + 1` (bỏ qua dấu `-`), nếu không vòng lặp dừng ngay và ta đọc ra chuỗi rỗng.

In [5]:
def read_number(s, i):
    """Đọc số bắt đầu tại s[i]. Trả về (giá trị, vị trí ngay sau số)."""
    j = i + 1                       # ký tự đầu là chữ số hoặc '-', luôn thuộc số
    while j < len(s) and s[j].isdigit():
        j += 1
    return int(s[i:j]), j

print(read_number("4(2)", 0))        # (4, 1)
print(read_number("-4(2)", 0))       # (-4, 2)
print(read_number("4(12)(305)", 2))  # (12, 4)
print(read_number("4(12)(305)", 6))  # (305, 9)
print(read_number("1073741824", 0))  # số lớn tới 2^30

(4, 1)
(-4, 2)
(12, 4)
(305, 9)
(1073741824, 10)


### 3.5. Đếm ngoặc để tìm cặp khớp nhau

Quy tắc: đi từ trái sang phải, gặp `(` thì `depth += 1`, gặp `)` thì `depth -= 1`. Cặp ngoặc mở tại `p` **khớp** với dấu `)` đầu tiên khiến `depth` quay về mức trước khi mở. Nhờ vậy ta biết "khối ngoặc" `( ... )` của một cây con kết thúc ở đâu, dù bên trong còn nhiều ngoặc lồng nhau.

In [6]:
s = "4(2(3)(1))(6(5))"
depth = 0
for i, c in enumerate(s):
    if c == "(":
        depth += 1
    elif c == ")":
        depth -= 1
    print(f"i={i:>2} ký tự {c!r:4} depth sau = {depth}")

i= 0 ký tự '4'  depth sau = 0
i= 1 ký tự '('  depth sau = 1
i= 2 ký tự '2'  depth sau = 1
i= 3 ký tự '('  depth sau = 2
i= 4 ký tự '3'  depth sau = 2
i= 5 ký tự ')'  depth sau = 1
i= 6 ký tự '('  depth sau = 2
i= 7 ký tự '1'  depth sau = 2
i= 8 ký tự ')'  depth sau = 1
i= 9 ký tự ')'  depth sau = 0
i=10 ký tự '('  depth sau = 1
i=11 ký tự '6'  depth sau = 1
i=12 ký tự '('  depth sau = 2
i=13 ký tự '5'  depth sau = 2
i=14 ký tự ')'  depth sau = 1
i=15 ký tự ')'  depth sau = 0


Quan sát: khối ngoặc đầu tiên mở ở `i=1` (`depth` từ 0 lên 1) và **đóng ở `i=9`** (lần đầu `depth` về 0). Bên trong khối đó, `s[2:9] = "2(3)(1)"` chính là cây con trái. Khối thứ hai mở ở `i=10` và đóng ở `i=15`.

Chỗ này sinh ra một cách giải trực tiếp (cách 1 bên dưới). Cũng có thể lưu vị trí các dấu `(` trong một **ngăn xếp** và khi gặp `)` thì `pop` ra để biết nó khớp với dấu `(` nào (cách 4).

### 3.6. Ngăn xếp (stack) bằng `list`

Ngăn xếp: **vào sau, ra trước** (LIFO). Trong Python dùng `list`: `append(x)` để đẩy vào, `pop()` để lấy phần tử trên cùng ra, `stack[-1]` để **xem** phần tử trên cùng mà không lấy ra.

In [7]:
stack = []
stack.append("a")
stack.append("b")
stack.append("c")
print(stack, "đỉnh =", stack[-1])
print("pop ->", stack.pop(), "| còn lại", stack)
print("pop ->", stack.pop(), "| còn lại", stack)
print("rỗng không?", not stack, "| sau pop cuối ->", stack.pop(), "| rỗng không?", not stack)

['a', 'b', 'c'] đỉnh = c
pop -> c | còn lại ['a', 'b']
pop -> b | còn lại ['a']
rỗng không? False | sau pop cuối -> a | rỗng không? True


Vì sao ngăn xếp hợp với bài này? Khi duyệt xâu, ta đang đi **xuống** từng cấp cây (gặp `(`) rồi **quay lên** (gặp `)`). Nút đang "mở" gần nhất luôn ở **đỉnh** ngăn xếp: đó đúng là kiểu vào-sau-ra-trước.

### 3.7. Đệ quy, `nonlocal` và giới hạn độ sâu đệ quy

Hàm tự gọi chính nó là **đệ quy**. Hàm lồng nhau (hàm trong hàm) có thể **đọc** biến của hàm ngoài; muốn **gán lại** thì phải khai báo `nonlocal`. Python giới hạn độ sâu đệ quy (mặc định khoảng `1000`) để tránh tràn bộ nhớ; chạm giới hạn sẽ báo `RecursionError`. Với cây dài tới `10^4` tầng (xâu `1(1(1(...)))` chỉ cần `3 * 10^4` ký tự) đệ quy có thể gặp rắc rối, vì thế một cách giải bằng ngăn xếp tường minh luôn an toàn.

In [8]:
def count_down(n):
    return 0 if n == 0 else 1 + count_down(n - 1)

def counter_demo():
    count = 0
    def bump():
        nonlocal count          # không có dòng này sẽ lỗi UnboundLocalError
        count += 1
    bump(); bump(); bump()
    return count

print(count_down(50), counter_demo())

old = sys.getrecursionlimit()
sys.setrecursionlimit(300)       # hạ giới hạn thật thấp để quan sát lỗi
try:
    count_down(1000)
except RecursionError:
    print("RecursionError: đệ quy quá sâu")
sys.setrecursionlimit(100000)    # nâng cao cho các thử nghiệm cây sâu phía dưới
print("giới hạn cũ:", old, "-> mới:", sys.getrecursionlimit())

50 3
RecursionError: đệ quy quá sâu
giới hạn cũ: 1000 -> mới: 100000


### 3.8. Bộ so sánh kết quả: `tree_to_list`, `tree2str`, `same_tree`

Ba hàm ở cell đầu là "thước đo" của notebook:

- `tree_to_list(root)` : in cây theo từng tầng để so với đáp án của đề.
- `tree2str(root)` : chuyển cây về xâu theo đúng định dạng; nếu `tree2str(str2tree(s)) == s` thì ta tin việc dựng đúng. Đây cũng là **oracle độc lập** (không dùng chung ý tưởng với các hàm dựng cây).
- `same_tree(a, b)` : hai cây giống hệt về hình dạng và giá trị.

Thử nhanh:

In [9]:
t1 = build_tree([4, 2, 6, 3, 1, 5])
t2 = build_tree([4, 2, 6, 3, 1, 5])
t3 = build_tree([4, 2, 6, 3, 1, None, 5])
print(same_tree(t1, t2), same_tree(t1, t3))
print(tree2str(t1), "|", tree2str(build_tree([1, None, 2])), "|", repr(tree2str(None)))

True False
4(2(3)(1))(6(5)) | 1()(2) | ''


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy dựng tay cây từ `s = "4(2(3)(1))(6(5))"` mà **không nghĩ đến code**, chỉ như một người đọc chữ:

1. Đọc `4`: tạo **gốc** `4`. Ta đang đứng ở nút `4`.
2. Gặp `(`: "đi xuống". Đây là cặp ngoặc **đầu tiên** của nút `4` nên ta sắp tạo con **trái** của `4`.
3. Đọc `2`: tạo nút `2`, treo vào `4.left`. Giờ đứng ở nút `2`.
4. Gặp `(`: đi xuống tiếp, cặp ngoặc đầu của `2` -> con trái. Đọc `3`: treo vào `2.left`.
5. Gặp `)`: xong cây con của `3`, **quay lên** nút `2`.
6. Gặp `(`: cặp ngoặc **thứ hai** của `2` -> con phải. Đọc `1`: treo vào `2.right`. Gặp `)`: quay lên `2`.
7. Gặp `)`: xong cây con của `2`, quay lên `4`.
8. Gặp `(`: cặp ngoặc **thứ hai** của `4` -> con phải. Đọc `6`, treo vào `4.right`; gặp `(`, đọc `5` treo vào `6.left`; gặp `)` quay lên `6`; gặp `)` quay lên `4`. Hết xâu.

Hai điều rút ra:

- **Ta cần nhớ "mình đang đứng ở nút nào" và "tổ tiên của nó"** để quay lên đúng chỗ: đó là một **ngăn xếp** (hoặc ngăn xếp gọi hàm của đệ quy).
- Với mỗi nút ta cần nhớ **đã mở bao nhiêu cặp ngoặc** (0, 1 hay 2) để biết nút mới là con trái hay con phải.

Bảng nhanh của quá trình tính tay (đường đi từ gốc tới nút hiện tại):

| Ký tự đọc | Hành động | Đường đi (gốc -> hiện tại) |
|---|---|---|
| `4` | tạo gốc | 4 |
| `(` | chuẩn bị con trái của 4 | 4 |
| `2` | `4.left = 2` | 4 -> 2 |
| `(` `3` | `2.left = 3` | 4 -> 2 -> 3 |
| `)` | quay lên | 4 -> 2 |
| `(` `1` | `2.right = 1` | 4 -> 2 -> 1 |
| `)` `)` | quay lên hai lần | 4 |
| `(` `6` | `4.right = 6` | 4 -> 6 |
| `(` `5` | `6.left = 5` | 4 -> 6 -> 5 |
| `)` `)` | quay lên hai lần | 4 |

## 5. Cách 1 — Brute Force / baseline: cắt xâu, đếm ngoặc, đệ quy trên xâu con

**Ý tưởng.** Bám sát ngữ pháp ở mục 3.2:

1. Nếu xâu rỗng -> trả `None`.
2. Tìm vị trí `p` của dấu `(` đầu tiên. Nếu **không có** `(` thì cả xâu là một số -> nút lá.
3. Ngược lại, `s[:p]` là giá trị gốc. Từ `p` quét bằng bộ đếm `depth` để tìm dấu `)` khớp với `(` tại `p`: được khối trái `s[p+1 : q]`.
4. Phần còn lại sau `q` (nếu không rỗng) là khối phải, có dạng `(...)`: lấy `rest[1:-1]`.
5. Gọi đệ quy cho hai xâu con.

Đây là cách **trực tiếp** nhất: chia để trị trên chính xâu. Phiên bản dưới đây là hàm demo; bản dạng class ở ngay sau.

In [10]:
def str2tree_brute(s):
    if not s:
        return None
    p = s.find("(")
    if p == -1:
        return TreeNode(int(s))                 # chỉ có một số -> nút lá
    root = TreeNode(int(s[:p]))
    depth = 0
    q = p
    for q in range(p, len(s)):                  # tìm dấu ')' khớp với '(' tại p
        if s[q] == "(":
            depth += 1
        elif s[q] == ")":
            depth -= 1
        if depth == 0:
            break
    root.left = str2tree_brute(s[p + 1:q])      # nội dung trong khối trái
    rest = s[q + 1:]                            # phần sau khối trái
    if rest:
        root.right = str2tree_brute(rest[1:-1]) # bỏ '(' đầu và ')' cuối của khối phải
    return root

for s, expected in examples:
    got = tree_to_list(str2tree_brute(s))
    print(repr(s), "->", got, "OK" if got == expected else "SAI")

'4(2(3)(1))(6(5))' -> [4, 2, 6, 3, 1, 5] OK
'4(2(3)(1))(6(5)(7))' -> [4, 2, 6, 3, 1, 5, 7] OK
'-4(2(3)(1))(6(5)(7))' -> [-4, 2, 6, 3, 1, 5, 7] OK


In [11]:
class Solution1:
    """Cách 1: cắt xâu + đếm ngoặc + đệ quy."""
    def str2tree(self, s: str) -> Optional[TreeNode]:
        if not s:
            return None
        p = s.find("(")
        if p == -1:
            return TreeNode(int(s))
        root = TreeNode(int(s[:p]))
        depth = 0
        q = p
        for q in range(p, len(s)):
            if s[q] == "(":
                depth += 1
            elif s[q] == ")":
                depth -= 1
            if depth == 0:
                break
        root.left = self.str2tree(s[p + 1:q])
        rest = s[q + 1:]
        if rest:
            root.right = self.str2tree(rest[1:-1])
        return root

print(tree_to_list(Solution1().str2tree("4(2(3)(1))(6(5)(7))")))
print(tree_to_list(Solution1().str2tree("")), tree_to_list(Solution1().str2tree("-4")))
print(tree_to_list(Solution1().str2tree("1()(2)")))

[4, 2, 6, 3, 1, 5, 7]
[] [-4]
[1, None, 2]


**Độ phức tạp.**

- Mỗi lần gọi ta **quét** khối trái bằng bộ đếm, và **cắt** (sao chép) các xâu con. Một ký tự nằm sâu `d` tầng bị quét/sao chép khoảng `d` lần (một lần cho mỗi tổ tiên chứa nó).
- Cây cân bằng (độ sâu ~ `log n`): khoảng `O(n log n)`.
- Cây **lệch** (một đường dài, độ sâu ~ `n/3`): mỗi tầng quét cả phần xâu còn lại -> **`O(n^2)`** với `n` tới `3 * 10^4`.
- Không gian: `O(n)` cho các xâu con tạm cộng với ngăn xếp gọi hàm.

**Nút thắt.** Ta đọc đi đọc lại cùng một ký tự nhiều lần (đếm ngoặc, cắt lát) chỉ để biết "khối này kết thúc ở đâu". Hãy **đo** xem chuyện đó tệ cỡ nào trên cây lệch bằng cách tăng gấp đôi kích thước:

In [12]:
def left_chain_string(depth):
    """Chuỗi cho cây chỉ toàn con trái, độ sâu depth: 1(1(1(...)))"""
    return "1(" * (depth - 1) + "1" + ")" * (depth - 1)

def measure(fn, s):
    t0 = time.perf_counter()
    fn(s)
    return (time.perf_counter() - t0) * 1000

print("depth | độ dài xâu | brute (ms)")
for d in [200, 400, 800, 1600]:
    s = left_chain_string(d)
    print(f"{d:>5} | {len(s):>10} | {measure(str2tree_brute, s):>9.1f}")

depth | độ dài xâu | brute (ms)
  200 |        598 |       4.9
  400 |       1198 |      19.6
  800 |       2398 |      88.2


 1600 |       4798 |     340.0


Khi `depth` **gấp đôi**, thời gian tăng gần **gấp bốn**: đúng dấu hiệu `O(n^2)`. Với `10^4` tầng, cách này sẽ rất chậm. Ta cần một cách mà **mỗi ký tự chỉ đọc một lần**.

## 6. Ý tưởng thuật toán / bước đột phá

Nhìn lại cách tính tay ở mục 4: ta **không hề quét lại** ký tự nào. Ta đi từ trái sang phải một lần, và chỉ cần biết ba thứ:

1. Ký tự hiện tại là gì: **số** (tạo nút mới), `(` (mở một cặp ngoặc cho nút đang đứng), hay `)` (đóng, quay lên).
2. **Nút đang đứng** là nút nào (nút "mở" gần nhất chưa đóng).
3. Nút đó **đã mở bao nhiêu cặp ngoặc** để biết nút mới là con trái (cặp đầu) hay con phải (cặp thứ hai).

Ba thứ này gói gọn trong một **ngăn xếp** các nút đang mở, mỗi phần tử kèm bộ đếm số ngoặc `(` đã mở. Đệ quy cũng làm đúng việc đó: ngăn xếp gọi hàm là ngăn xếp các nút đang mở, còn con trỏ `i` dùng chung cho cả hàm chạy dần về phải.

**Bước đột phá (một câu):** *đừng cắt xâu để tìm khối con; hãy đi một lượt, dùng ngăn xếp (hoặc đệ quy) để nhớ tổ tiên, và `(` = xuống, `)` = lên.*

Ta có ba cách thực hiện ý tưởng này: đệ quy với con trỏ (cách 2), ngăn xếp tường minh (cách 3), và biến thể tính sẵn "dấu `(` nào khớp với dấu `)` nào" (cách 4).

## 7. Các cách tiếp theo

### Cách 2 — Đệ quy một lượt với con trỏ (recursive descent parser)

**Giải thích từng đoạn.**

- Dùng một con trỏ `i` (biến `nonlocal`) cho biết "đã đọc tới đâu". Mọi hàm đệ quy dùng chung `i`, và mỗi hàm **đẩy `i` về phía trước** đúng bằng số ký tự nó tiêu thụ.
- `parse_node()`: đang đứng ở ký tự đầu của một số. Đọc số (mục 3.4), tạo nút, rồi: nếu `s[i] == "("` thì gọi `parse_child()` để được con trái; sau đó nếu `s[i] == "("` **lần nữa** thì gọi `parse_child()` để được con phải.
- `parse_child()`: đang đứng ở `(`. Bỏ qua `(`. Nếu ngay sau là `)` thì con là `None` (cặp ngoặc rỗng), ngược lại gọi `parse_node()`. Cuối cùng bỏ qua `)`.

Hai hàm gọi nhau theo đúng cấu trúc ngữ pháp ở mục 3.2: mỗi "luật" của ngữ pháp thành một hàm. Đây là kỹ thuật **recursive descent**.

Demo từng đoạn (ta dựng thử một nút lá rồi nút có hai con):

In [13]:
def demo_descent(s):
    n = len(s)
    i = 0

    def parse_node():
        nonlocal i
        j = i + 1                       # đoạn 1: đọc số (có thể âm, nhiều chữ số)
        while j < n and s[j].isdigit():
            j += 1
        node = TreeNode(int(s[i:j]))
        i = j
        if i < n and s[i] == "(":       # đoạn 2: cặp ngoặc thứ nhất -> trái
            node.left = parse_child()
            if i < n and s[i] == "(":   # đoạn 3: cặp ngoặc thứ hai -> phải
                node.right = parse_child()
        return node

    def parse_child():
        nonlocal i
        i += 1                          # bỏ qua '('
        child = None if s[i] == ")" else parse_node()
        i += 1                          # bỏ qua ')'
        return child

    return parse_node() if n else None

print(tree_to_list(demo_descent("7")))
print(tree_to_list(demo_descent("7(8)(9)")))
print(tree_to_list(demo_descent("4(2(3)(1))(6(5))")))

[7]
[7, 8, 9]
[4, 2, 6, 3, 1, 5]


**Code nộp LeetCode (cách 2):**

In [14]:
class Solution2:
    """Cách 2: đệ quy với con trỏ i dùng chung."""
    def str2tree(self, s: str) -> Optional[TreeNode]:
        n = len(s)
        i = 0

        def parse_node():
            nonlocal i
            j = i + 1
            while j < n and s[j].isdigit():
                j += 1
            node = TreeNode(int(s[i:j]))
            i = j
            if i < n and s[i] == "(":
                node.left = parse_child()
                if i < n and s[i] == "(":
                    node.right = parse_child()
            return node

        def parse_child():
            nonlocal i
            i += 1                      # bỏ '('
            child = None if s[i] == ")" else parse_node()
            i += 1                      # bỏ ')'
            return child

        return parse_node() if n else None

for s, expected in examples:
    assert tree_to_list(Solution2().str2tree(s)) == expected
print("cách 2 qua 3 ví dụ")

cách 2 qua 3 ví dụ


**Độ phức tạp (cách 2).** Mỗi ký tự được đọc đúng một lần (con trỏ `i` chỉ tiến, không lùi) và không cắt xâu -> **Time `O(n)`**. Space: ngăn xếp đệ quy sâu bằng chiều cao cây `h` -> `O(h)` (xấu nhất `O(n)` với cây lệch) cộng `O(n)` cho các nút tạo ra. Lưu ý: cây lệch `10^4` tầng cần nâng giới hạn đệ quy.

### Cách 3 — Một lượt quét với ngăn xếp tường minh (khuyên dùng khi nộp)

**Giải thích từng đoạn.**

- `stack` chứa các phần tử `[node, opened]`: nút đang mở và **số dấu `(` đã gặp** ở nút đó (0, 1 hoặc 2).
- Gặp `(`: `stack[-1][1] += 1`. Từ đây nút trên đỉnh đã mở thêm một cặp ngoặc; số này cho biết nút **kế tiếp** (nếu có) là con trái (`opened == 1`) hay con phải (`opened == 2`).
- Gặp chữ số hoặc `-`: đọc cả số, tạo `node`. Nếu ngăn xếp rỗng thì đây là **gốc**. Ngược lại gắn vào cha (nút ở đỉnh): `left` nếu `opened == 1`, `right` nếu `opened == 2`. Rồi đẩy `[node, 0]` vào ngăn xếp.
- Gặp `)`: thường nghĩa là nút trên đỉnh đã xong -> `pop`. **Ngoại lệ:** nếu ký tự ngay trước là `(` thì đó là cặp `()` rỗng, chưa hề đẩy nút nào, nên **không** `pop`.

Demo đọc từng đoạn nhỏ:

In [15]:
def demo_stack(s):
    stack, root = [], None
    i, n = 0, len(s)
    while i < n:
        c = s[i]
        if c == "(":
            stack[-1][1] += 1                  # đoạn A: nút trên đỉnh mở thêm một ngoặc
            i += 1
        elif c == ")":
            if s[i - 1] != "(":                # đoạn B: đóng cây con có nút -> quay lên
                stack.pop()
            i += 1
        else:                                  # đoạn C: số -> tạo nút
            j = i + 1
            while j < n and s[j].isdigit():
                j += 1
            node = TreeNode(int(s[i:j]))
            if stack:
                parent, opened = stack[-1]
                if opened == 1:
                    parent.left = node
                else:
                    parent.right = node
            else:
                root = node
            stack.append([node, 0])
            i = j
    return root

print(tree_to_list(demo_stack("7(8)(9)")))
print(tree_to_list(demo_stack("7()(9)")))
print(tree_to_list(demo_stack("-4(2(3)(1))(6(5)(7))")))

[7, 8, 9]
[7, None, 9]
[-4, 2, 6, 3, 1, 5, 7]


**Code nộp LeetCode (cách 3) - đây cũng là code chính ở mục 9:**

In [16]:
class Solution3:
    """Cách 3: ngăn xếp tường minh, đọc mỗi ký tự một lần."""
    def str2tree(self, s: str) -> Optional[TreeNode]:
        stack = []                       # phần tử: [nút, số dấu '(' đã mở ở nút đó]
        root = None
        i, n = 0, len(s)
        while i < n:
            c = s[i]
            if c == "(":
                stack[-1][1] += 1
                i += 1
            elif c == ")":
                if s[i - 1] != "(":
                    stack.pop()
                i += 1
            else:
                j = i + 1
                while j < n and s[j].isdigit():
                    j += 1
                node = TreeNode(int(s[i:j]))
                if stack:
                    parent, opened = stack[-1]
                    if opened == 1:
                        parent.left = node
                    else:
                        parent.right = node
                else:
                    root = node
                stack.append([node, 0])
                i = j
        return root

for s, expected in examples:
    assert tree_to_list(Solution3().str2tree(s)) == expected
print("cách 3 qua 3 ví dụ")

cách 3 qua 3 ví dụ


**Độ phức tạp (cách 3).** Con trỏ `i` chỉ tiến, mỗi ký tự xử lý `O(1)` (đọc số tổng cộng cũng tối đa `n`) -> **Time `O(n)`**. Ngăn xếp chứa tối đa `h` phần tử (các nút trên đường từ gốc) -> Space phụ `O(h)`, không đệ quy nên **không bao giờ `RecursionError`**.

### Cách 4 — Tính sẵn cặp ngoặc khớp nhau, rồi đệ quy theo khoảng chỉ số

**Giải thích từng đoạn.**

- **Bước 1 (tiền xử lý):** quét xâu một lần với ngăn xếp các vị trí `(`. Gặp `)` thì `pop` ra vị trí `p` của dấu `(` tương ứng và ghi `match[p] = i`. Từ đây hỏi "dấu `(` tại `p` đóng ở đâu?" chỉ tốn `O(1)`.
- **Bước 2:** hàm `build(l, r)` dựng cây cho **đoạn xâu `s[l:r]`** (không cắt xâu, chỉ truyền hai chỉ số). Đọc số từ `l` tới dấu `(` (hoặc hết đoạn). Nếu có `(` tại `j`: con trái là `build(j+1, match[j])`. Dấu `(` thứ hai (nếu có) nằm ở `k = match[j] + 1` -> con phải là `build(k+1, match[k])`.

Khác với cách 1: không đếm ngoặc lặp lại (đã tính sẵn một lần), không sao chép xâu.

Demo bước 1:

In [17]:
def match_brackets(s):
    match = [-1] * len(s)
    stack = []
    for i, c in enumerate(s):
        if c == "(":
            stack.append(i)
        elif c == ")":
            match[stack.pop()] = i
    return match

s = "4(2(3)(1))(6(5))"
m = match_brackets(s)
print({p: m[p] for p in range(len(s)) if m[p] != -1})

{1: 9, 3: 5, 6: 8, 10: 15, 12: 14}


Kết quả: `1 -> 9` (khối trái của `4`), `3 -> 5`, `6 -> 8`, `10 -> 15`, `12 -> 14`: chính là các cặp ta đã tìm bằng mắt ở mục 3.5.

**Code nộp LeetCode (cách 4):**

In [18]:
class Solution4:
    """Cách 4: tính sẵn dấu ngoặc khớp nhau, đệ quy trên khoảng [l, r)."""
    def str2tree(self, s: str) -> Optional[TreeNode]:
        n = len(s)
        match = [0] * n
        stack = []
        for i, c in enumerate(s):
            if c == "(":
                stack.append(i)
            elif c == ")":
                match[stack.pop()] = i

        def build(l, r):                  # dựng cây từ s[l:r]
            if l >= r:
                return None
            j = l + 1
            while j < r and s[j].isdigit():
                j += 1
            node = TreeNode(int(s[l:j]))
            if j < r:                     # s[j] == "(": có con trái
                node.left = build(j + 1, match[j])
                k = match[j] + 1
                if k < r:                 # s[k] == "(": có con phải
                    node.right = build(k + 1, match[k])
            return node

        return build(0, n)

for s, expected in examples:
    assert tree_to_list(Solution4().str2tree(s)) == expected
print("cách 4 qua 3 ví dụ")

cách 4 qua 3 ví dụ


**Độ phức tạp (cách 4).** Bước 1 là `O(n)`. Bước 2: mỗi ký tự được đọc đúng một lần ở đúng một lời gọi `build` -> `O(n)`. Tổng **Time `O(n)`**. Space: mảng `match` cỡ `n` và ngăn xếp tạm -> `O(n)`, cộng ngăn xếp đệ quy `O(h)`. Cách này tốn bộ nhớ hơn cách 3 nhưng tách bạch rõ hai việc: "tìm cặp ngoặc" và "dựng cây".

**Tóm tắt bốn cách:** (1) cắt xâu + đếm ngoặc -> `O(n^2)` xấu nhất; (2) đệ quy với con trỏ -> `O(n)`; (3) ngăn xếp tường minh -> `O(n)`, an toàn nhất; (4) tính sẵn ngoặc khớp -> `O(n)`.

## 8. TRACE từng bước

### 8.1. Trace cách 3 (ngăn xếp) trên `"4(2(3)(1))(6(5))"`

Hàm in từng ký tự, hành động, và nội dung ngăn xếp sau bước đó (viết `giá trị:số ngoặc đã mở`).

| Cột | Ý nghĩa |
|---|---|
| `i`, `c` | vị trí và ký tự đang đọc |
| `hành động` | tạo nút / mở ngoặc / pop |
| `stack` | các nút đang mở, từ gốc (trái) tới nút hiện tại (phải) |

In [19]:
def trace_stack(s):
    stack, root = [], None
    i, n = 0, len(s)
    step = 0
    while i < n:
        c = s[i]
        if c == "(":
            stack[-1][1] += 1
            act = f"mở ngoặc thứ {stack[-1][1]} của {stack[-1][0].val}"
            width = 1
        elif c == ")":
            if s[i - 1] != "(":
                done = stack.pop()
                act = f"pop {done[0].val} (xong cây con)"
            else:
                act = "() rỗng: không pop"
            width = 1
        else:
            j = i + 1
            while j < n and s[j].isdigit():
                j += 1
            node = TreeNode(int(s[i:j]))
            if stack:
                parent, opened = stack[-1]
                side = "left" if opened == 1 else "right"
                setattr(parent, side, node)
                act = f"tạo {node.val}, gắn vào {parent.val}.{side}, push"
            else:
                root = node
                act = f"tạo {node.val} làm GỐC, push"
            stack.append([node, 0])
            width = j - i
        step += 1
        view = [f"{nd.val}:{k}" for nd, k in stack]
        print(f"i={i:>2} c={s[i:i+width]!r:>5} | {act:<34} | stack = {view}")
        i += width
    return root

r = trace_stack("4(2(3)(1))(6(5))")
print("kết quả:", tree_to_list(r))

i= 0 c=  '4' | tạo 4 làm GỐC, push                | stack = ['4:0']
i= 1 c=  '(' | mở ngoặc thứ 1 của 4               | stack = ['4:1']
i= 2 c=  '2' | tạo 2, gắn vào 4.left, push        | stack = ['4:1', '2:0']
i= 3 c=  '(' | mở ngoặc thứ 1 của 2               | stack = ['4:1', '2:1']
i= 4 c=  '3' | tạo 3, gắn vào 2.left, push        | stack = ['4:1', '2:1', '3:0']
i= 5 c=  ')' | pop 3 (xong cây con)               | stack = ['4:1', '2:1']
i= 6 c=  '(' | mở ngoặc thứ 2 của 2               | stack = ['4:1', '2:2']
i= 7 c=  '1' | tạo 1, gắn vào 2.right, push       | stack = ['4:1', '2:2', '1:0']
i= 8 c=  ')' | pop 1 (xong cây con)               | stack = ['4:1', '2:2']
i= 9 c=  ')' | pop 2 (xong cây con)               | stack = ['4:1']
i=10 c=  '(' | mở ngoặc thứ 2 của 4               | stack = ['4:2']
i=11 c=  '6' | tạo 6, gắn vào 4.right, push       | stack = ['4:2', '6:0']
i=12 c=  '(' | mở ngoặc thứ 1 của 6               | stack = ['4:2', '6:1']
i=13 c=  '5' | tạo 5, gắn vào 6.left, p

Quan sát: ngăn xếp luôn đúng bằng **đường đi từ gốc tới nút hiện tại** (đúng như bảng tính tay ở mục 4). Khi gặp `)` ta `pop` một nút = quay lên một cấp. Nút `4` ở cuối vẫn còn trong ngăn xếp vì nó là gốc, không có `)` đóng riêng.

### 8.2. Trace cách 2 (đệ quy với con trỏ)

In ra lúc **vào** và **ra** mỗi hàm, thụt lề theo độ sâu đệ quy, và cho thấy `i` nhảy tới đâu.

In [20]:
def trace_descent(s):
    n = len(s)
    i = 0

    def parse_node(depth):
        nonlocal i
        pad = "  " * depth
        j = i + 1
        while j < n and s[j].isdigit():
            j += 1
        node = TreeNode(int(s[i:j]))
        print(f"{pad}parse_node: đọc số {node.val} ở s[{i}:{j}] -> i = {j}")
        i = j
        if i < n and s[i] == "(":
            print(f"{pad}  gặp '(' ở i={i}: dựng con TRÁI của {node.val}")
            node.left = parse_child(depth + 1)
            if i < n and s[i] == "(":
                print(f"{pad}  gặp '(' ở i={i}: dựng con PHẢI của {node.val}")
                node.right = parse_child(depth + 1)
        print(f"{pad}xong nút {node.val}, i = {i}")
        return node

    def parse_child(depth):
        nonlocal i
        i += 1
        if s[i] == ")":
            child = None
        else:
            child = parse_node(depth)
        i += 1
        return child

    return parse_node(0) if n else None

r = trace_descent("4(2(3)(1))(6(5))")
print("kết quả:", tree_to_list(r))

parse_node: đọc số 4 ở s[0:1] -> i = 1
  gặp '(' ở i=1: dựng con TRÁI của 4
  parse_node: đọc số 2 ở s[2:3] -> i = 3
    gặp '(' ở i=3: dựng con TRÁI của 2
    parse_node: đọc số 3 ở s[4:5] -> i = 5
    xong nút 3, i = 5
    gặp '(' ở i=6: dựng con PHẢI của 2
    parse_node: đọc số 1 ở s[7:8] -> i = 8
    xong nút 1, i = 8
  xong nút 2, i = 9
  gặp '(' ở i=10: dựng con PHẢI của 4
  parse_node: đọc số 6 ở s[11:12] -> i = 12
    gặp '(' ở i=12: dựng con TRÁI của 6
    parse_node: đọc số 5 ở s[13:14] -> i = 14
    xong nút 5, i = 14
  xong nút 6, i = 15
xong nút 4, i = 16
kết quả: [4, 2, 6, 3, 1, 5]


Quan sát: `i` luôn **tăng**, không bao giờ lùi. Hàm `parse_node` của `2` kết thúc ở `i = 10` (ngay sau `)` thứ hai), và đó đúng là chỗ `4` thấy dấu `(` thứ hai để dựng con phải.

### 8.3. Trace cách 1 (cắt xâu) để thấy chỗ lãng phí

In mỗi lần gọi: xâu con, giá trị gốc, khối trái, khối phải.

In [21]:
def trace_brute(s, depth=0):
    pad = "  " * depth
    if not s:
        print(f"{pad}s={s!r} -> None")
        return None
    p = s.find("(")
    if p == -1:
        print(f"{pad}s={s!r} -> lá {int(s)}")
        return TreeNode(int(s))
    root = TreeNode(int(s[:p]))
    d = 0
    q = p
    for q in range(p, len(s)):
        d += (s[q] == "(") - (s[q] == ")")
        if d == 0:
            break
    left_s, rest = s[p + 1:q], s[q + 1:]
    right_s = rest[1:-1] if rest else ""
    print(f"{pad}s={s!r}: gốc={root.val}, trái={left_s!r}, phải={right_s!r}")
    root.left = trace_brute(left_s, depth + 1)
    if rest:
        root.right = trace_brute(right_s, depth + 1)
    return root

r = trace_brute("4(2(3)(1))(6(5))")
print("kết quả:", tree_to_list(r))

s='4(2(3)(1))(6(5))': gốc=4, trái='2(3)(1)', phải='6(5)'
  s='2(3)(1)': gốc=2, trái='3', phải='1'
    s='3' -> lá 3
    s='1' -> lá 1
  s='6(5)': gốc=6, trái='5', phải=''
    s='5' -> lá 5
kết quả: [4, 2, 6, 3, 1, 5]


Để ý chuỗi `"3"`, `"1"`, `"5"` chỉ là vài ký tự, nhưng các xâu như `"2(3)(1)"` bị **sao chép và quét lại** ở mỗi tầng. Ở cây sâu, lượng việc lặp này làm cách 1 chậm như đã đo ở mục 5.

## 9. Code nộp LeetCode

Dùng **cách 3** (ngăn xếp tường minh): chạy `O(n)`, không đệ quy nên không sợ cây sâu `10^4` tầng, và chịu được cả `()` rỗng.

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def str2tree(self, s: str) -> Optional[TreeNode]:
        stack = []                       # [nút, số dấu '(' đã mở ở nút đó]
        root = None
        i, n = 0, len(s)
        while i < n:
            c = s[i]
            if c == "(":
                stack[-1][1] += 1
                i += 1
            elif c == ")":
                if s[i - 1] != "(":      # "()" rỗng thì không có nút để pop
                    stack.pop()
                i += 1
            else:
                j = i + 1
                while j < n and s[j].isdigit():
                    j += 1
                node = TreeNode(int(s[i:j]))
                if stack:
                    parent, opened = stack[-1]
                    if opened == 1:
                        parent.left = node
                    else:
                        parent.right = node
                else:
                    root = node
                stack.append([node, 0])
                i = j
        return root
```

Chạy lại đúng đoạn code trên trong notebook:

In [22]:
class Solution:
    def str2tree(self, s: str) -> Optional[TreeNode]:
        stack = []                       # [nút, số dấu '(' đã mở ở nút đó]
        root = None
        i, n = 0, len(s)
        while i < n:
            c = s[i]
            if c == "(":
                stack[-1][1] += 1
                i += 1
            elif c == ")":
                if s[i - 1] != "(":      # "()" rỗng thì không có nút để pop
                    stack.pop()
                i += 1
            else:
                j = i + 1
                while j < n and s[j].isdigit():
                    j += 1
                node = TreeNode(int(s[i:j]))
                if stack:
                    parent, opened = stack[-1]
                    if opened == 1:
                        parent.left = node
                    else:
                        parent.right = node
                else:
                    root = node
                stack.append([node, 0])
                i = j
        return root

# Mô phỏng cách LeetCode gọi: Solution().str2tree(s) rồi so với đáp án
for s, expected in examples:
    got = tree_to_list(Solution().str2tree(s))
    print(repr(s), "->", got, "OK" if got == expected else "SAI")
    assert got == expected

'4(2(3)(1))(6(5))' -> [4, 2, 6, 3, 1, 5] OK
'4(2(3)(1))(6(5)(7))' -> [4, 2, 6, 3, 1, 5, 7] OK
'-4(2(3)(1))(6(5)(7))' -> [-4, 2, 6, 3, 1, 5, 7] OK


## 10. Vì sao thuật toán đúng

Gọi một nút là **"đang mở"** nếu ta đã đọc số của nó nhưng chưa đọc xong mọi cặp ngoặc của nó. Ta chứng minh bằng **bất biến vòng lặp** (cách 3):

> **Bất biến:** trước khi đọc ký tự `s[i]`, ngăn xếp chứa đúng các nút đang mở, từ gốc (đáy) đến nút sâu nhất (đỉnh), mỗi nút kèm số cặp ngoặc đã **mở** ở nút đó; và mọi nút đã được tạo đều gắn đúng vị trí vào cha.

- **Khởi đầu:** ngăn xếp rỗng, chưa tạo nút nào. Bất biến đúng.
- **Gặp số:** theo ngữ pháp, một số xuất hiện ở đầu xâu (gốc) hoặc **ngay sau một `(`**. Trường hợp sau, nút ở đỉnh vừa mở cặp ngoặc thứ `opened`: `opened = 1` -> đây là con trái, `opened = 2` -> con phải, và ta gắn đúng như vậy. Nút mới "đang mở" nên được đẩy lên đỉnh.
- **Gặp `(`:** nút ở đỉnh bắt đầu thêm một cặp ngoặc: tăng `opened` đúng 1. Không có nút mới, không có nút nào đóng.
- **Gặp `)`:** nó đóng cặp ngoặc gần nhất chưa đóng, tức là cặp mở của nút ở đỉnh. Nếu bên trong cặp có một cây con (ký tự ngay trước không phải `(`) thì **nút gốc của cây con đó** nằm ở đỉnh và vừa được đọc xong toàn bộ -> `pop`. Nếu bên trong rỗng (`()`) thì chưa hề có nút nào được đẩy vào cho cặp này -> không `pop`. Bất biến vẫn giữ.
- **Kết thúc:** ngăn xếp chỉ còn gốc, và mọi nút đã được gắn đúng chỗ, nên cây dựng ra là cây mà xâu mô tả.

Về **thứ tự trái/phải**: ngữ pháp quy định cặp ngoặc thứ nhất là trái, thứ hai là phải; ta dùng bộ đếm `opened` đúng theo thứ tự xuất hiện nên không thể nhầm.

Về **cách 2 và 4**: cùng là cài đặt trực tiếp của ngữ pháp "Cây := số [ (Cây) [ (Cây) ] ]" bằng đệ quy: tính đúng đắn đi theo quy nạp trên độ dài xâu (xâu con trong ngoặc luôn ngắn hơn xâu cha). Với cách 2, mỗi hàm tiêu thụ **đúng** các ký tự thuộc phần của nó và để `i` trỏ ngay sau, nên hàm cha tiếp tục đúng chỗ.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)` (tối đa `3 * 10^4`) và `h` là chiều cao của cây (tối đa khoảng `n / 3`).

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Cắt xâu + đếm ngoặc | `O(n log n)` trung bình, **`O(n^2)`** xấu nhất | `O(n)` xâu tạm + `O(h)` đệ quy | đọc lặp ký tự |
| 2. Đệ quy + con trỏ | `O(n)` | `O(h)` đệ quy | cần nâng recursion limit khi `h` lớn |
| 3. Ngăn xếp tường minh | `O(n)` | `O(h)` | **không đệ quy**, an toàn nhất |
| 4. Tính sẵn ngoặc khớp | `O(n)` | `O(n)` mảng `match` + `O(h)` | tách bạch hai bước |

Cộng thêm `O(n)` cho bản thân cây kết quả (số nút `<= n`), không tính là "phụ" vì đó là đầu ra.

Vì sao cách 3 là `O(n)`? Mỗi vòng `while` tiến `i` ít nhất 1; riêng nhánh đọc số tiến `j - i` bước và số chữ số của mọi số cộng lại không quá `n`. Mỗi nút được `append` một lần và `pop` tối đa một lần.

## 12. Bộ test

### 12.1. Ví dụ của đề và các ví dụ tự đặt

Tạo một danh sách các cách để chạy cùng một bộ test (mỗi cách nhận xâu, trả cây):

In [23]:
METHODS = {
    "1-cat-xau": Solution1().str2tree,
    "2-de-quy": Solution2().str2tree,
    "3-ngan-xep": Solution3().str2tree,
    "4-ngoac-khop": Solution4().str2tree,
    "nop": Solution().str2tree,
}

def run_case(s, expected_list):
    for name, fn in METHODS.items():
        got = tree_to_list(fn(s))
        assert got == expected_list, (name, s, got, expected_list)

cases = [
    ("4(2(3)(1))(6(5))", [4, 2, 6, 3, 1, 5]),
    ("4(2(3)(1))(6(5)(7))", [4, 2, 6, 3, 1, 5, 7]),
    ("-4(2(3)(1))(6(5)(7))", [-4, 2, 6, 3, 1, 5, 7]),
]
for s, e in cases:
    run_case(s, e)
print("ví dụ của đề: OK")

ví dụ của đề: OK


### 12.2. Edge case

- Xâu rỗng -> `None` (list rỗng).
- Một nút; số âm; số nhiều chữ số; số lớn `2^30 = 1073741824`; `0`; `-0` không có trong đề nên bỏ qua.
- Chỉ có con trái (chuỗi lệch trái), kiểu `"1(2(3))"`.
- Chỉ có con phải, bằng định dạng `()`: `"1()(2()(3))"`.
- Ba mức, nút âm ở lá, v.v.

In [24]:
edge = [
    ("", []),
    ("0", [0]),
    ("5", [5]),
    ("-5", [-5]),
    ("1073741824", [1073741824]),
    ("-1073741824(1073741824)(-1073741824)", [-1073741824, 1073741824, -1073741824]),
    ("12(345)(-6789)", [12, 345, -6789]),
    ("1(2(3))", [1, 2, None, 3]),
    ("1(2)(3)", [1, 2, 3]),
    ("1()(2)", [1, None, 2]),
    ("1()(2()(3))", [1, None, 2, None, 3]),
    ("1(2(3(4)))(5)", [1, 2, 5, 3, None, None, None, 4]),
    ("-1(-2(-3))(-4(-5)(-6))", [-1, -2, -4, -3, None, -5, -6]),
]
for s, e in edge:
    run_case(s, e)
    print(f"{s!r:42} -> {e}")
print("edge case: OK")

''                                         -> []
'0'                                        -> [0]
'5'                                        -> [5]
'-5'                                       -> [-5]
'1073741824'                               -> [1073741824]
'-1073741824(1073741824)(-1073741824)'     -> [-1073741824, 1073741824, -1073741824]
'12(345)(-6789)'                           -> [12, 345, -6789]
'1(2(3))'                                  -> [1, 2, None, 3]
'1(2)(3)'                                  -> [1, 2, 3]
'1()(2)'                                   -> [1, None, 2]
'1()(2()(3))'                              -> [1, None, 2, None, 3]
'1(2(3(4)))(5)'                            -> [1, 2, 5, 3, None, None, None, 4]
'-1(-2(-3))(-4(-5)(-6))'                   -> [-1, -2, -4, -3, None, -5, -6]
edge case: OK


### 12.3. Phản ví dụ: những input làm các hàm "gần đúng" sai

Các xâu dưới đây được chọn để **bắt** những lỗi hay gặp (mục 13): số nhiều chữ số, số âm, con phải, cây sâu hai nhánh. Ta kiểm tra lại: cách đúng vượt qua, và xâu sau khi tuần tự hóa phải **trùng lại chính nó** (round trip).

In [25]:
tricky = [
    "10(20)(30)",             # số nhiều chữ số: đọc từng ký tự sẽ tách "10" thành 1 và 0
    "-1(-2)(-3)",             # số âm
    "1(2)(3(4)(5))",          # con phải có hai con
    "1(2(3)(4))(5(6)(7))",    # cây đầy đủ
    "1(2(3(4(5))))",          # lệch trái
    "1()(2()(3()(4)))",       # lệch phải (kiểu 606)
]
for s in tricky:
    t = Solution().str2tree(s)
    assert tree2str(t) == s, s              # oracle độc lập: tuần tự hóa phải ra đúng xâu ban đầu
    for fn in METHODS.values():
        assert same_tree(fn(s), t)
    print(f"{s:24} -> {tree_to_list(t)}")
print("phản ví dụ: OK")

10(20)(30)               -> [10, 20, 30]
-1(-2)(-3)               -> [-1, -2, -3]
1(2)(3(4)(5))            -> [1, 2, 3, None, None, 4, 5]
1(2(3)(4))(5(6)(7))      -> [1, 2, 5, 3, 4, 6, 7]
1(2(3(4(5))))            -> [1, 2, None, 3, None, 4, None, 5]
1()(2()(3()(4)))         -> [1, None, 2, None, 3, None, 4]
phản ví dụ: OK


### 12.4. Đối chiếu bằng cây ngẫu nhiên (seed cố định)

Cách làm: sinh **cây ngẫu nhiên**, **tuần tự hóa** bằng `tree2str` (hàm hoàn toàn khác với các hàm dựng cây), rồi dựng lại bằng mỗi cách và so sánh với cây gốc bằng `same_tree`. Như vậy "đáp án" đến từ cây gốc chứ không từ chính lời giải.

In [26]:
def random_tree(rng, n, lo, hi):
    """Cây ngẫu nhiên đúng n nút (n >= 1), giá trị trong [lo, hi]."""
    root = TreeNode(rng.randint(lo, hi))
    slots = [(root, "left"), (root, "right")]
    for _ in range(n - 1):
        k = rng.randrange(len(slots))
        slots[k], slots[-1] = slots[-1], slots[k]
        parent, side = slots.pop()
        child = TreeNode(rng.randint(lo, hi))
        setattr(parent, side, child)
        slots.append((child, "left"))
        slots.append((child, "right"))
    return root

rng = random.Random(536)
count = 0
for _ in range(600):
    n = rng.randint(1, 30)
    lo, hi = rng.choice([(0, 9), (-9, 9), (-1000, 1000), (-(2**30), 2**30)])
    original = random_tree(rng, n, lo, hi)
    s = tree2str(original)
    assert len(s) <= 3 * 10**4
    for name, fn in METHODS.items():
        assert same_tree(fn(s), original), (name, s)
    count += 1
print("đã đối chiếu", count, "cây ngẫu nhiên với", len(METHODS), "cách: OK")

đã đối chiếu 600 cây ngẫu nhiên với 5 cách: OK


Thêm một lượt cây **lớn hơn** (nhiều trăm nút) cho các cách có độ phức tạp tốt, và một lượt cây **kiểu bài đề** (chỉ chứa con trái; nếu nút chỉ có con phải thì ta bỏ qua định dạng `()` bằng cách đổi con phải thành con trái) để chắc rằng định dạng "gốc" của đề vẫn đúng:

In [27]:
def normalize_no_empty(root):
    """Biến mọi nút chỉ-có-con-phải thành chỉ-có-con-trái, để xâu không chứa '()' (đúng đề gốc)."""
    stack = [root]
    while stack:
        x = stack.pop()
        if x.left is None and x.right is not None:
            x.left, x.right = x.right, None
        for c in (x.left, x.right):
            if c:
                stack.append(c)
    return root

rng = random.Random(2024)
for _ in range(100):
    original = random_tree(rng, rng.randint(100, 600), -99, 99)
    s = tree2str(original)
    for name in ["2-de-quy", "3-ngan-xep", "4-ngoac-khop", "nop"]:
        assert same_tree(METHODS[name](s), original), name

for _ in range(200):
    original = normalize_no_empty(random_tree(rng, rng.randint(1, 40), -50, 50))
    s = tree2str(original)
    assert "()" not in s
    for name, fn in METHODS.items():
        assert same_tree(fn(s), original), (name, s)
print("cây lớn và cây đúng định dạng đề: OK")

cây lớn và cây đúng định dạng đề: OK


### 12.5. Test hiệu năng và cây sâu (giới hạn `3 * 10^4` ký tự)

- Cây lệch trái `10^4` tầng, xâu `3 * 10^4` ký tự: **ngăn xếp** và **đệ quy** (đã nâng `recursionlimit`) phải chạy trong chốc lát; cách 1 sẽ chậm nên không đưa vào.
- Cây ngẫu nhiên nhiều nút nhất có thể trong giới hạn xâu.

In [28]:
deep = left_chain_string(10000)
print("độ dài xâu:", len(deep))
for name in ["2-de-quy", "3-ngan-xep", "4-ngoac-khop", "nop"]:
    t0 = time.perf_counter()
    t = METHODS[name](deep)
    ms = (time.perf_counter() - t0) * 1000
    assert tree2str(t) == deep
    print(f"{name:14} {ms:7.1f} ms | chiều cao = 10000 | round trip OK")

độ dài xâu: 29998
2-de-quy           8.7 ms | chiều cao = 10000 | round trip OK
3-ngan-xep         8.8 ms | chiều cao = 10000 | round trip OK
4-ngoac-khop       9.3 ms | chiều cao = 10000 | round trip OK
nop                8.5 ms | chiều cao = 10000 | round trip OK


In [29]:
rng = random.Random(7)
big = random_tree(rng, 6000, 0, 9)
s_big = tree2str(big)
print("độ dài xâu big:", len(s_big), "<= 30000:", len(s_big) <= 30000)
for name in ["2-de-quy", "3-ngan-xep", "4-ngoac-khop", "nop"]:
    t0 = time.perf_counter()
    t = METHODS[name](s_big)
    ms = (time.perf_counter() - t0) * 1000
    assert same_tree(t, big)
    print(f"{name:14} {ms:7.1f} ms")
t0 = time.perf_counter()
assert same_tree(METHODS["1-cat-xau"](s_big), big)
print(f"{'1-cat-xau':14} {(time.perf_counter() - t0) * 1000:7.1f} ms (cây ngẫu nhiên nên nông, brute vẫn ổn)")

độ dài xâu big: 20122 <= 30000: True
2-de-quy           3.5 ms
3-ngan-xep         7.3 ms
4-ngoac-khop       4.9 ms
nop                4.9 ms
1-cat-xau         17.2 ms (cây ngẫu nhiên nên nông, brute vẫn ổn)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đọc từng ký tự thay vì đọc cả số

**Code sai:** coi mỗi chữ số là một nút.

In [30]:
def wrong_single_char(s):
    stack, root = [], None
    for c in s:
        if c == "(":
            stack[-1][1] += 1
        elif c == ")":
            stack.pop()
        else:
            node = TreeNode(int(c))            # SAI: c chỉ là MỘT ký tự
            if stack:
                parent, opened = stack[-1]
                if opened == 1:
                    parent.left = node
                else:
                    parent.right = node
            else:
                root = node
            stack.append([node, 0])
    return root

print("đúng :", tree_to_list(Solution().str2tree("12(3)")))
print("sai  :", tree_to_list(wrong_single_char("12(3)")))
try:
    wrong_single_char("-4(2)")
except ValueError as e:
    print("với '-4(2)' ->", type(e).__name__, ":", e)

đúng : [12, 3]
sai  : [1, None, 2, 3]
với '-4(2)' -> ValueError : invalid literal for int() with base 10: '-'


**Vì sao sai:** số có thể **nhiều chữ số** (`12`) và **âm** (`-4`). Đọc từng ký tự làm `12` thành hai nút `1` và `2`, còn `'-'` không phải số nên `int('-')` lỗi.

**Code đúng:** dùng hai con trỏ `i`, `j` để đọc **cả số** `s[i:j]` (mục 3.4), nhớ bắt đầu `j = i + 1` để dấu `-` được tính vào số.

### Lỗi 2: Luôn gán vào `left` (mất con phải)

In [31]:
def wrong_always_left(s):
    stack, root = [], None
    i, n = 0, len(s)
    while i < n:
        c = s[i]
        if c == "(":
            i += 1
        elif c == ")":
            stack.pop()
            i += 1
        else:
            j = i + 1
            while j < n and s[j].isdigit():
                j += 1
            node = TreeNode(int(s[i:j]))
            if stack:
                stack[-1].left = node          # SAI: luôn gán trái, con thứ hai đè con thứ nhất
            else:
                root = node
            stack.append(node)
            i = j
    return root

print("đúng :", tree_to_list(Solution().str2tree("4(2)(6)")))
print("sai  :", tree_to_list(wrong_always_left("4(2)(6)")), "(con 2 bị 6 ghi đè, mất con phải)")

đúng : [4, 2, 6]
sai  : [4, 6] (con 2 bị 6 ghi đè, mất con phải)


**Vì sao sai:** nút `6` là cặp ngoặc **thứ hai** của `4` nên phải vào `right`, nhưng code gán `left` lần nữa nên đè mất `2`.

**Code đúng:** nhớ **số ngoặc đã mở** ở mỗi nút (`opened`): `1` -> trái, `2` -> phải. Cách 2 làm điều này tự nhiên: gọi `parse_child()` lần thứ nhất cho trái, lần thứ hai cho phải.

### Lỗi 3: Quên `pop` khi gặp `)`

In [32]:
def wrong_no_pop(s):
    stack, root = [], None
    i, n = 0, len(s)
    while i < n:
        c = s[i]
        if c == "(":
            stack[-1][1] += 1
            i += 1
        elif c == ")":
            i += 1                              # SAI: không pop, không "quay lên"
        else:
            j = i + 1
            while j < n and s[j].isdigit():
                j += 1
            node = TreeNode(int(s[i:j]))
            if stack:
                parent, opened = stack[-1]
                if opened == 1:
                    parent.left = node
                else:
                    parent.right = node
            else:
                root = node
            stack.append([node, 0])
            i = j
    return root

print("đúng :", tree_to_list(Solution().str2tree("4(2(3)(1))(6)")))
try:
    print("sai  :", tree_to_list(wrong_no_pop("4(2(3)(1))(6)")))
except Exception as e:
    print("sai  : lỗi", type(e).__name__)

đúng : [4, 2, 6, 3, 1]
sai  : [4, 2, None, 3, None, 1, None, 6]


**Vì sao sai:** sau khi đọc xong `3`, ta phải quay lên `2` để `(1)` thành con **phải của `2`**; không `pop` thì ngăn xếp vẫn để `3` trên đỉnh và `1` bị gắn nhầm vào `3`. Cứ thế các nút treo thành một **chuỗi dài**.

**Code đúng:** mỗi dấu `)` đóng một cây con có nút -> `stack.pop()` (trừ cặp `()` rỗng).

### Lỗi 4: Cách đệ quy quên bỏ qua `)`

In [33]:
def wrong_no_skip(s):
    n = len(s)
    i = 0
    def parse_node():
        nonlocal i
        j = i + 1
        while j < n and s[j].isdigit():
            j += 1
        node = TreeNode(int(s[i:j]))
        i = j
        if i < n and s[i] == "(":
            i += 1
            node.left = parse_node()
            # SAI: quên i += 1 để bỏ qua ')' của con trái
            if i < n and s[i] == "(":
                i += 1
                node.right = parse_node()
                # SAI: quên bỏ ')' của con phải
        return node
    return parse_node() if n else None

try:
    print("sai  :", tree_to_list(wrong_no_skip("4(2)(6)")))
except Exception as e:
    print("sai  : lỗi", type(e).__name__, "-", e)
print("đúng :", tree_to_list(Solution2().str2tree("4(2)(6)")))

sai  : [4, 2]
đúng : [4, 2, 6]


**Vì sao sai:** sau khi dựng xong con trái, con trỏ vẫn đứng ở `)`; điều kiện `s[i] == "("` sai nên con phải **không bao giờ được dựng** (hoặc đọc sai chỗ). Mọi hàm đệ quy dùng con trỏ chung phải **tiêu thụ hết** phần xâu của mình, kể cả dấu `)` đóng.

**Code đúng:** trong `parse_child`, sau khi dựng xong con luôn `i += 1` để nhảy qua `)`.

### Lỗi 5: Quên trường hợp xâu rỗng và đọc `int("")`

In [34]:
def wrong_empty(s):
    stack, root = [], None
    j = 1
    while j < len(s) and s[j].isdigit():
        j += 1
    return TreeNode(int(s[:j]))     # với s = "" -> int("") lỗi

try:
    wrong_empty("")
except ValueError as e:
    print("sai  : ValueError:", e)
print("đúng :", tree_to_list(Solution().str2tree("")), "(cây rỗng -> list rỗng)")

# lỗi liên quan: bắt đầu j = i thay vì i + 1 khiến '-' bị bỏ ngoài số
def wrong_minus(s, i=0):
    j = i
    while j < len(s) and s[j].isdigit():
        j += 1
    return s[i:j]

print("sai (j=i):", repr(wrong_minus("-4(2)")), "-> int('') sẽ lỗi")
print("đúng     :", read_number("-4(2)", 0))

sai  : ValueError: invalid literal for int() with base 10: ''
đúng : [] (cây rỗng -> list rỗng)
sai (j=i): '' -> int('') sẽ lỗi
đúng     : (-4, 2)


**Vì sao sai:** đề cho phép `s.length = 0`; cần kiểm tra `if not s: return None` ngay đầu hàm. Và nếu `j` bắt đầu từ `i` thì dấu `-` làm vòng lặp dừng ngay (vì `"-".isdigit()` là `False`), lấy được chuỗi rỗng.

**Code đúng:** `if not s: return None`; và `j = i + 1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Nhận dạng:** xâu có **ngoặc lồng nhau** biểu diễn cấu trúc đệ quy -> nghĩ tới **ngăn xếp** hoặc **đệ quy** (recursive descent). Cụm từ "string -> tree/nested structure" là tín hiệu.
- Quy tắc vàng: `(` = đi xuống một cấp, `)` = quay lên một cấp. Ngăn xếp chính là **đường đi từ gốc đến nút hiện tại**.
- Tránh cắt xâu trong đệ quy nếu có thể: truyền **chỉ số** hoặc dùng **con trỏ dùng chung**, khi đó mỗi ký tự được đọc đúng một lần.
- Cây có thể sâu tới `10^4`; nếu ngôn ngữ/hệ chấm không cho đệ quy sâu, chọn **ngăn xếp tường minh**.
- Đọc số: nhớ cả **dấu âm** và **nhiều chữ số**. Mẫu chuẩn: `j = i + 1; while j < n and s[j].isdigit(): j += 1`.
- Bài họ hàng: LeetCode 606 (cây -> xâu, chiều ngược lại), 297 (tuần tự hóa/giải tuần tự hóa cây), 385 (Mini Parser, ngoặc vuông lồng nhau), 394 (Decode String), 224/772 (máy tính cơ bản, ngoặc và ngăn xếp), 726 (Number of Atoms), 20 (Valid Parentheses).
- **Mẹo kiểm tra:** tuần tự hóa lại cây dựng được bằng hàm độc lập (`tree2str`) và so với xâu ban đầu: round trip đúng thì gần như chắc chắn đúng.

## 15. Biến thể

### 15.1. Tuần tự hóa ngược lại (LeetCode 606): cây -> xâu

Chính hàm `tree2str` mà ta dùng làm oracle. Phiên bản đệ quy ngắn gọn để so sánh với bản lặp:

In [35]:
def tree2str_recursive(root):
    if root is None:
        return ""
    s = str(root.val)
    if root.left is None and root.right is None:
        return s
    s += "(" + tree2str_recursive(root.left) + ")"
    if root.right is not None:
        s += "(" + tree2str_recursive(root.right) + ")"
    return s

rng = random.Random(15)
for _ in range(200):
    t = random_tree(rng, rng.randint(1, 40), -20, 20)
    assert tree2str_recursive(t) == tree2str(t)
print(tree2str_recursive(build_tree([1, 2, 3, None, 4])))

1(2()(4))(3)


### 15.2. Kiểm tra xâu có đúng định dạng không (valid encoding)

Trước khi dựng cây, kiểm tra: ngoặc cân bằng, mỗi nút không quá hai cặp ngoặc, sau `(` phải là số hoặc `)`, v.v. Dùng một ngăn xếp đếm số cặp ngoặc đã mở ở mỗi nút:

In [36]:
def is_valid_encoding(s):
    if s == "":
        return True
    i, n = 0, len(s)
    stack = []                       # mỗi phần tử: số cặp ngoặc đã mở của nút đang mở
    expect_number = True             # ở đầu và ngay sau '(' (nếu không phải '()') cần một số
    seen_root = False
    while i < n:
        c = s[i]
        if c == "(":
            if not stack or stack[-1] >= 2 or expect_number:
                return False
            stack[-1] += 1
            expect_number = True
            i += 1
        elif c == ")":
            if expect_number and s[i - 1] != "(":
                return False
            if s[i - 1] != "(":
                stack.pop()
            elif not stack:
                return False
            expect_number = False
            i += 1
        elif c == "-" or c.isdigit():
            if not expect_number:
                return False
            j = i + 1 if c == "-" else i
            if j >= n or not s[j].isdigit():
                return False
            while j < n and s[j].isdigit():
                j += 1
            if not stack and seen_root:
                return False
            seen_root = True
            stack.append(0)
            expect_number = False
            i = j
        else:
            return False
    return len(stack) == 1 and not expect_number

good = ["7", "7(8)", "7(8)(9)", "-4(2(3)(1))(6(5)(7))", "1()(2)", ""]
bad = ["(7)", "7(8)(9)(10)", "7(", "7)", "7(8", "7-", "7(8))", "(", "78(9", "7 (8)"]
print([is_valid_encoding(x) for x in good])
print([is_valid_encoding(x) for x in bad])
assert all(is_valid_encoding(x) for x in good)
assert not any(is_valid_encoding(x) for x in bad)

[True, True, True, True, True, True]
[False, False, False, False, False, False, False, False, False, False]


### 15.3. Cây N-ary: `"1(2(5))(3)(4)"` (mỗi nút nhiều con)

Cùng kỹ thuật ngăn xếp, chỉ khác là không còn "trái/phải" mà là **danh sách con**. Ta dựng cây dạng `(giá trị, [các con])`:

In [37]:
def parse_nary(s):
    root = None
    stack = []                          # chỉ chứa các nút đang mở: [giá trị, danh sách con]
    i, n = 0, len(s)
    while i < n:
        c = s[i]
        if c == "(":
            i += 1
        elif c == ")":
            stack.pop()
            i += 1
        else:
            j = i + 1
            while j < n and s[j].isdigit():
                j += 1
            node = [int(s[i:j]), []]
            if stack:
                stack[-1][1].append(node)
            else:
                root = node
            stack.append(node)
            i = j
    return root

print(parse_nary("1(2(5))(3)(4)"))

[1, [[2, [[5, []]]], [3, []], [4, []]]]


### 15.4. Đổi sang kiểu list theo tầng và ngược lại

Kết hợp hai bước: xâu -> cây -> `tree_to_list` (đã dùng suốt notebook), và `build_tree` + `tree2str` cho chiều ngược lại. Hai hàm này là cầu nối giữa **ba cách biểu diễn cây**: xâu ngoặc, list theo tầng, và đối tượng `TreeNode`.

In [38]:
for s in ["4(2(3)(1))(6(5))", "1()(2)", "9(8(7(6)))"]:
    t = Solution().str2tree(s)
    lst = tree_to_list(t)
    back = tree2str(build_tree(lst))
    print(f"{s:20} -> list {lst} -> xâu {back}")
    assert back == s

4(2(3)(1))(6(5))     -> list [4, 2, 6, 3, 1, 5] -> xâu 4(2(3)(1))(6(5))
1()(2)               -> list [1, None, 2] -> xâu 1()(2)
9(8(7(6)))           -> list [9, 8, None, 7, None, 6] -> xâu 9(8(7(6)))


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Cắt xâu + đếm ngoặc | Chia để trị trực tiếp trên xâu con | `O(n log n)` / `O(n^2)` xấu nhất | `O(n)` | để hiểu bài, xâu ngắn |
| 2. Đệ quy + con trỏ | Mỗi luật ngữ pháp là một hàm, con trỏ chung | `O(n)` | `O(h)` đệ quy | cây nông, muốn code giống ngữ pháp |
| 3. Ngăn xếp tường minh | `(` xuống, `)` lên; stack = đường đi | `O(n)` | `O(h)` | **khuyên dùng**: nhanh, không lo đệ quy sâu |
| 4. Tính sẵn ngoặc khớp | Mảng `match`, rồi đệ quy theo khoảng | `O(n)` | `O(n)` | khi cần biết cặp ngoặc nhiều lần (nhiều truy vấn) |

Kết luận: chọn **cách 3** khi nộp bài; cách 2 khi viết nhanh và muốn dễ đọc; cách 4 khi cần tra cứu cặp ngoặc.

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán đầu ra của hai dòng sau rồi chạy:

In [39]:
print(tree_to_list(Solution().str2tree("1(2)(3)")))
print(tree_to_list(Solution().str2tree("10(-2(7))(5)")))

[1, 2, 3]
[10, -2, 5, 7]


**Bài 2 (tính tay).** Hãy viết ra xâu tuần tự hóa (theo định dạng 606: nếu chỉ có con phải thì con trái là `()`) của cây `[1,2,3,None,4]`. Gán vào `ans2` (đang là `None`).

In [40]:
ans2 = None      # TODO: điền xâu, ví dụ "1(...)(...)"

**Bài 3 (điền TODO).** Viết `max_nesting(s)` trả về **độ sâu ngoặc lớn nhất** của xâu (độ sâu lớn nhất của `depth` ở mục 3.5). Không cần dựng cây. Ví dụ `"4(2(3)(1))(6(5))"` -> `2`; `"7"` -> `0`.

In [41]:
def max_nesting(s):
    # TODO: dùng bộ đếm depth, cập nhật best = max(best, depth) khi gặp '('
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_str2tree(s)` theo cách 3 (ngăn xếp). Chịu được xâu rỗng và số âm nhiều chữ số.

In [42]:
def my_str2tree(s):
    # TODO
    pass

**Bài 5 (mini challenge).** Viết `count_leaves(s)`: đếm số **nút lá** của cây mà xâu mô tả, **không dựng cây**. Gợi ý: một số là lá nếu ký tự ngay sau số đó **không phải** `(`. Sau đó viết `leaf_values(s)` trả về **danh sách giá trị các lá theo thứ tự từ trái sang phải** của xâu (thứ tự xuất hiện).

In [43]:
def count_leaves(s):
    # TODO
    pass

def leaf_values(s):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: dùng ngữ pháp "số, (trái), (phải)": ở xâu thứ hai, `10` là gốc, `(-2(7))` là trái, `(5)` là phải.
- Bài 2: vẽ cây ra giấy: gốc `1`, con trái `2`, con phải `3`, và `2` có con **phải** `4`.
- Bài 3: duyệt ký tự; `(` thì tăng depth, `)` thì giảm depth.
- Bài 4: ngăn xếp các nút đang mở, nhớ số ngoặc đã mở của mỗi nút.
- Bài 5: tìm các vị trí bắt đầu số; nhìn ký tự ngay sau chữ số cuối của số.

**Mức 2 (vừa):**
- Bài 2: nút `2` chỉ có con phải nên viết `2()(4)`; nút `3` là lá nên `3`. Ghép lại: `1(` + xâu của 2 + `)(` + xâu của 3 + `)`.
- Bài 3: `depth = best = 0`; mỗi `(` làm `depth += 1; best = max(best, depth)`.
- Bài 4: khi gặp số, `opened == 1` -> trái, `opened == 2` -> phải; gặp `)` thì pop (trừ `()` rỗng).
- Bài 5: dùng hàm `read_number(s, i)` ở mục 3.4 để lấy `(giá trị, j)`; số là lá nếu `j == len(s)` hoặc `s[j] != "("`.

**Mức 3 (gần đáp án):**
- Bài 2: `1(2()(4))(3)`.
- Bài 3: `best = max(best, depth)` đặt **ngay sau** `depth += 1`.
- Bài 5: vòng `while i < n`: nếu `s[i]` là `(` hoặc `)` thì `i += 1`, ngược lại `v, j = read_number(s, i)`; nếu `j >= n or s[j] != "("` thì `v` là lá; sau đó `i = j`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [44]:
# Bài 1: [1, 2, 3] và [10, -2, 5, 7]
assert tree_to_list(Solution().str2tree("1(2)(3)")) == [1, 2, 3]
assert tree_to_list(Solution().str2tree("10(-2(7))(5)")) == [10, -2, 5, 7]

# Bài 2: nút 2 chỉ có con phải 4 -> "2()(4)"
ans2 = "1(2()(4))(3)"
assert tree2str(build_tree([1, 2, 3, None, 4])) == ans2
assert same_tree(Solution().str2tree(ans2), build_tree([1, 2, 3, None, 4]))
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [45]:
# Bài 3
def max_nesting(s):
    depth = best = 0
    for c in s:
        if c == "(":
            depth += 1
            best = max(best, depth)
        elif c == ")":
            depth -= 1
    return best

assert max_nesting("4(2(3)(1))(6(5))") == 2
assert max_nesting("7") == 0
assert max_nesting("") == 0
assert max_nesting("1(2)(3)") == 1
rng = random.Random(19)
for _ in range(100):
    t = random_tree(rng, rng.randint(1, 40), 0, 9)
    s = tree2str(t)
    # độ sâu ngoặc lớn nhất = chiều cao cây tính theo số cạnh
    h, stack = 0, [(t, 0)]
    while stack:
        x, d = stack.pop()
        h = max(h, d)
        for c in (x.left, x.right):
            if c:
                stack.append((c, d + 1))
    assert max_nesting(s) == h
print("Bài 3 OK")

Bài 3 OK


In [46]:
# Bài 4
def my_str2tree(s):
    stack, root = [], None
    i, n = 0, len(s)
    while i < n:
        c = s[i]
        if c == "(":
            stack[-1][1] += 1
            i += 1
        elif c == ")":
            if s[i - 1] != "(":
                stack.pop()
            i += 1
        else:
            j = i + 1
            while j < n and s[j].isdigit():
                j += 1
            node = TreeNode(int(s[i:j]))
            if stack:
                parent, opened = stack[-1]
                if opened == 1:
                    parent.left = node
                else:
                    parent.right = node
            else:
                root = node
            stack.append([node, 0])
            i = j
    return root

rng = random.Random(4)
for _ in range(200):
    t = random_tree(rng, rng.randint(1, 40), -(2**30), 2**30)
    assert same_tree(my_str2tree(tree2str(t)), t)
assert my_str2tree("") is None
print("Bài 4 OK")

Bài 4 OK


In [47]:
# Bài 5
def count_leaves(s):
    return len(leaf_values(s))

def leaf_values(s):
    out = []
    i, n = 0, len(s)
    while i < n:
        if s[i] in "()":
            i += 1
        else:
            v, j = read_number(s, i)
            if j >= n or s[j] != "(":
                out.append(v)
            i = j
    return out

assert leaf_values("4(2(3)(1))(6(5))") == [3, 1, 5]
assert leaf_values("-7") == [-7]
assert leaf_values("") == []
assert count_leaves("4(2(3)(1))(6(5)(7))") == 4
rng = random.Random(55)
for _ in range(200):
    t = random_tree(rng, rng.randint(1, 40), -9, 9)
    s = tree2str(t)
    expected, stack = [], [t]
    # lá theo thứ tự trái -> phải: duyệt tiền tự, trái trước
    while stack:
        x = stack.pop()
        if x.left is None and x.right is None:
            expected.append(x.val)
        if x.right:
            stack.append(x.right)
        if x.left:
            stack.append(x.left)
    assert leaf_values(s) == expected
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template ngăn xếp (khuyên dùng):**

```python
class Solution:
    def str2tree(self, s: str) -> Optional[TreeNode]:
        stack, root = [], None          # stack: [nút, số '(' đã mở]
        i, n = 0, len(s)
        while i < n:
            c = s[i]
            if c == "(":
                stack[-1][1] += 1; i += 1
            elif c == ")":
                if s[i - 1] != "(":
                    stack.pop()
                i += 1
            else:                       # số (có thể âm, nhiều chữ số)
                j = i + 1
                while j < n and s[j].isdigit():
                    j += 1
                node = TreeNode(int(s[i:j]))
                if stack:
                    p, k = stack[-1]
                    if k == 1: p.left = node
                    else: p.right = node
                else:
                    root = node
                stack.append([node, 0])
                i = j
        return root
```

**Template đệ quy với con trỏ (cách 2):**

```python
i = 0
def parse_node():
    nonlocal i
    j = i + 1
    while j < n and s[j].isdigit(): j += 1
    node = TreeNode(int(s[i:j])); i = j
    if i < n and s[i] == "(":
        node.left = parse_child()
        if i < n and s[i] == "(":
            node.right = parse_child()
    return node
def parse_child():
    nonlocal i
    i += 1                              # bỏ '('
    child = None if s[i] == ")" else parse_node()
    i += 1                              # bỏ ')'
    return child
```

**Checklist trước khi submit:**

- [ ] Xâu rỗng `""` -> trả `None`?
- [ ] Đọc **cả số** (nhiều chữ số, dấu `-`), `j` bắt đầu từ `i + 1`?
- [ ] Cặp ngoặc **đầu** -> `left`, cặp **thứ hai** -> `right` (đếm `opened` hoặc gọi lần lượt)?
- [ ] Gặp `)` có `pop` / có `i += 1` bỏ qua `)` (và không `pop` nhầm cho `()` rỗng)?
- [ ] Mỗi ký tự đọc **một lần** (không cắt xâu lặp lại gây `O(n^2)`)?
- [ ] Cây sâu `10^4` tầng: dùng ngăn xếp tường minh (hoặc nâng recursion limit)?
- [ ] Đã thử: một nút, số âm, số nhiều chữ số, chỉ con trái, cây đầy đủ, `"1()(2)"`?
- [ ] Đã tuần tự hóa lại cây dựng được và so với xâu gốc (round trip)?